<a id="top"></a>

# Demo: self-critique — the good, the bad, the sycophantic

```yaml
title: "Demo: self-critique — the good, the bad, the sycophantic"
keywords: self-critique, sycophancy, adversarial framing, two-step critique, revision, teacher demo
estimated duration: 12
```

> **Lesson:** L06. Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L06/demos_or_activities.md). Makes **live** Claude calls through `potato_llm` — set `ANTHROPIC_API_KEY` before running. If the neutral critique catches the error on the first try, run it again — sycophancy doesn't show up on every run, so a second pass is worth it. Clear outputs before committing.
>
> Here's the line to hold onto: self-critique is a **sampling technique, not a correctness oracle**. A critic with no new information rubber-stamps the answer.

## Contents

- [1. Setup](#1-setup)
- [2. The first (often wrong) answer](#2-the-first-often-wrong-answer)
- [3. Neutral self-check — watch for sycophancy](#3-neutral-self-check--watch-for-sycophancy)
- [4. Adversarial framing — inject a different prior](#4-adversarial-framing--inject-a-different-prior)
- [5. Takeaways](#5-takeaways)

## 1. Setup

You'll use a prompt with a misleading surface (the classic bat-and-ball) that often tempts a plausible-but-wrong first answer. Below you'll set up helpers for that first answer and for a critique pass.

In [ ]:
from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

QUESTION = (
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost? Answer with a dollar amount."
)

client: PotatoLLMClient = AnthropicClient()


async def first_answer() -> str:
    """The cold first pass — no reasoning scaffold, so the trap often springs."""
    reply = await client.achat([Message.user(QUESTION)], max_tokens=80, temperature=0.0)
    return reply.text.strip()


async def critique(answer: str, framing: str) -> str:
    """Feed the first answer back for review under a given framing (two-step critique)."""
    messages = [
        Message.user(QUESTION),
        Message.assistant(answer),
        Message.user(framing),
    ]
    reply = await client.achat(messages, max_tokens=300, temperature=0.0)
    return reply.text.strip()


print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. The first (often wrong) answer

In [ ]:
answer = await first_answer()
print("first answer:", answer)  # the trap answer is $0.10

[↑ Back to top](#top)

## 3. Neutral self-check — watch for sycophancy

Same model, neutral framing, looking at its own answer. Roughly half the time you'll see it **agree with a wrong answer** — that's the failure mode worth watching for here: sycophancy.

In [ ]:
print(await critique(answer, "Review your answer above. Is it correct?"))

[↑ Back to top](#top)

## 4. Adversarial framing — inject a different prior

Same model, but now you're telling it to *find the flaw*. Notice what's really driving the higher catch rate: it isn't new knowledge — it's a shifted prior over what a good answer looks like.

In [ ]:
print(
    await critique(
        answer,
        "You are a skeptical reviewer. Find the flaw in the answer above and produce a "
        "corrected version. Show the algebra.",
    )
)

[↑ Back to top](#top)

## 5. Takeaways

- **Sycophancy** is the failure to remember: a same-model, same-prompt critic tends to agree with itself, right or wrong.
- A critic with **no new information** is weak. Adversarial framing, a different model, or retrieved evidence all *inject* something to disagree with.
- A **different model** as critic (e.g. Haiku 4.5) is the next lever — revisited rigorously in L13 (model power).
- This was a **two-step** critique (separate calls). A single-prompt inline critique is cheaper but weaker — it shares the context that produced the error.

[↑ Back to top](#top)